# Qwen2.5-VL

In [ ]:
import io
import matplotlib.pyplot as plt
import pandas as pd
import torch
from PIL import Image
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
from qwen_vl_utils import process_vision_info

# 1. Generate sample time-series data (e.g., IoT Sensor Stream)
data = {
    'time': range(24),
    'sensor_value': [12, 14, 13, 15, 28, 45, 52, 48, 30, 22, 19, 18, 
                     20, 22, 25, 24, 29, 46, 60, 55, 40, 28, 18, 14]
}
df = pd.DataFrame(data)

In [ ]:
# 2. Render time-series into a chart image (Modality Bridge)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(df['time'], df['sensor_value'], marker='o', color='teal', label='Sensor Load')
ax.set_title("24-Hour Industrial Sensor Stream")
ax.set_xlabel("Time (Hours)")
ax.set_ylabel("Reading Magnitude")
ax.grid(True)

image_path = "timeseries_chart.png"
plt.savefig(image_path, bbox_inches='tight')
plt.close()

In [ ]:
# 3. Load local Qwen2-VL-7B-Instruct
# model_id = "Qwen/Qwen2-VL-7B-Instruct"
# model_id = "Qwen/Qwen2.5-VL-7B-Instruct"

model_id = "Qwen/Qwen2.5-VL-3B-Instruct"

print(f"Loading local model: {model_id}...")

# Load processor and model ( utilizing half-precision float16 for memory efficiency )
processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct")

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id, 
    torch_dtype="auto", 
    device_map="auto"
)

Loading local model: Qwen/Qwen2.5-VL-3B-Instruct...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


In [12]:
from huggingface_hub import scan_cache_dir

# Scans your default Hugging Face cache
hf_cache = scan_cache_dir()

for repo in hf_cache.repos:
    if "Qwen2.5-VL" in repo.repo_id:
        print(f"Found Model: {repo.repo_id}")
        print(f"Storage Size: {repo.size_on_disk_str}")
        print(f"Path: {repo.repo_path}")

Found Model: Qwen/Qwen2.5-VL-3B-Instruct
Storage Size: 7.5G
Path: /home/ubuntu/.cache/huggingface/hub/models--Qwen--Qwen2.5-VL-3B-Instruct
Found Model: Qwen/Qwen2.5-VL-7B-Instruct
Storage Size: 16.6G
Path: /home/ubuntu/.cache/huggingface/hub/models--Qwen--Qwen2.5-VL-7B-Instruct


In [13]:
# 4. Construct messages using Qwen's standard chat template format
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image_path},
            {
                "type": "text", 
                "text": "Analyze this time series chart. Identify the specific hours where major spikes occur, and explain the overall trend sequence."
            },
        ],
    }
]

In [14]:
# 5. Prepare inputs using Qwen processor utils
text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt"
).to("cuda" if torch.cuda.is_available() else "cpu")

In [15]:
# 6. Generate inference response
print("Running inference with Qwen2.5-VL...")
with torch.no_grad():
    generated_ids = model.generate(**inputs, max_new_tokens=200)

# Trim input prompt tokens from output sequence
generated_ids_trimmed = [
    out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)[0]

print("\n--- Qwen2.5-VL Time-Series Analysis Result ---")
print(output_text)

Running inference with Qwen2.5-VL...

--- Qwen2.5-VL Time-Series Analysis Result ---
The 24-hour industrial sensor stream shows a series of peaks and troughs over time. The major spikes occur at approximately:

- **5 hours**: The reading magnitude reaches its highest point around 5 hours.
- **18 hours**: Another significant spike occurs around 18 hours.

The overall trend sequence can be described as follows:

- **Early Morning (0-5 hours)**: The readings start low and gradually increase.
- **Mid-Morning to Early Afternoon (5-10 hours)**: There is a noticeable rise in readings, peaking around 7-8 hours.
- **Late Afternoon to Early Evening (10-15 hours)**: The readings drop significantly, reaching a low point around 12-13 hours.
- **Evening to Night (15-20 hours)**: There is another rise in readings, peaking around 19-20 hours.
- **Mid-Night to Early Morning (2
